In [ ]:
import re
import pandas as pd
from cleantext import clean

df = pd.read_csv(r"\E-c\2018-E-c-En-test-gold.txt", sep="\t")
df.head()

In [ ]:
df.drop(columns=['ID'], inplace=True)
df.head()

In [ ]:
df.rename(columns={'Tweet': 'text', 'anger': 0, 'anticipation': 1, 'disgust': 2, 'fear': 3, 'joy': 4, 'love': 5,
                   'optimism': 6, 'pessimism': 7, 'sadness': 8, 'surprise': 9, 'trust': 10}, inplace=True)
df.head()

In [ ]:
def clean_text(text):
    # 1. Use cleantext to fix Unicode + transliterate
    text = clean(text,
                 fix_unicode=True,
                 to_ascii=True,
                 lower=False,
                 no_emoji=True,
                 no_urls=True,
                 no_currency_symbols=False)

    # 2. Remove stray currency symbols NOT followed by a number
    text = re.sub(r'([$€£₹])(?!\d)', '', text)

    # 3. Remove unwanted punctuation excluding . , ! ? $ % & #
    text = re.sub(r'[{}\[\]<>*`~/\\\^_|+=\-—…;:(\)]', '', text)

    # 4. Remove consecutive hyphens (replace with a single hyphen)
    text = re.sub(r'-+', '-', text)

    # 5. Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['text'] = df['text'].apply(clean_text)
df.head()

In [ ]:
# Replace usernames with [NAME]
df['text'] = df['text'].astype(str).apply(
    lambda x: re.sub(r'@\w+', '[NAME]', x)
)

In [ ]:
# Remove hashtags but keep the text
df['text'] = df['text'].astype(str).apply(
    lambda x: re.sub(r'(?<!\w)#(\w+)', r'\1', x)
)

In [ ]:
df.info()

In [ ]:
df.to_csv(r"path", index=False)

Check Train Data Distribution

In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv(r"path")
df.head()

In [ ]:
label_map = {
    0: "anger",
    1: "anticipation",
    2: "disgust",
    3: "fear",
    4: "joy",
    5: "love",
    6: "optimism",
    7: "pessimism",
    8: "sadness",
    9: "surprise",
    10: "trust"
}


label_columns = [str(i) for i in range(11)]

label_sums = df[label_columns].sum()

label_sums.index = label_sums.index.astype(int)
label_sums.index = label_sums.index.map(label_map)

fig = px.bar(
    x=label_sums.index,
    y=label_sums.values,
    text=label_sums.values,
    color=label_map.values(),
    labels={'x': 'Labels', 'y': 'Count'},
    title='Distribution of Labels in SemEval Dataset'
)

fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(height=500, width=700, showlegend=False, xaxis_tickangle=-45, plot_bgcolor='white',
                      xaxis=dict(showline=True, linewidth=2, linecolor='gray', mirror='ticks'),
    yaxis=dict(showline=True, linewidth=2, linecolor='gray', mirror='ticks'),)
fig.show()
